# 05 · 脉冲网络去雾骨架

LIF 神经元 + 代理梯度 + 时间步累积；输出对时间维平均。

In [ ]:
import torch
import torch.nn as nn


class LIFSpike(torch.autograd.Function):
    @staticmethod
    def forward(ctx, u, threshold):
        ctx.save_for_backward(u, threshold)
        return (u >= threshold).float()

    @staticmethod
    def backward(ctx, grad_out):
        u, threshold = ctx.saved_tensors
        grad = grad_out / (1.0 + torch.abs(u - threshold)) ** 2
        return grad, None


class LIFNeuron(nn.Module):
    def __init__(self, dim, init_tau=0.25, init_vth=0.25):
        super().__init__()
        self.tau = nn.Parameter(torch.tensor(init_tau))
        self.vth = nn.Parameter(torch.tensor(init_vth))
        self.mem = None

    def reset(self):
        self.mem = None

    def forward(self, x):
        if self.mem is None:
            self.mem = torch.zeros_like(x)
        self.mem = self.tau * self.mem + (1 - self.tau) * x
        spike = LIFSpike.apply(self.mem, self.vth)
        self.mem = self.mem - spike * self.vth
        return spike


class SpikingConvBlock(nn.Module):
    def __init__(self, dim, timesteps=4):
        super().__init__()
        self.timesteps = timesteps
        self.conv = nn.Conv2d(dim, dim, 3, padding=1, padding_mode="reflect")
        self.lif = LIFNeuron(dim)

    def forward(self, x):
        self.lif.reset()
        out = 0
        for _ in range(self.timesteps):
            out = out + self.lif(self.conv(x))
        return out / self.timesteps


class SpikingDehazeNet(nn.Module):
    def __init__(self, dim=24, n_blocks=4, timesteps=4):
        super().__init__()
        self.stem = nn.Conv2d(3, dim, 3, padding=1)
        self.blocks = nn.ModuleList(
            [SpikingConvBlock(dim, timesteps) for _ in range(n_blocks)])
        self.head = nn.Conv2d(dim, 3, 3, padding=1)

    def forward(self, x):
        feat = self.stem(x)
        for blk in self.blocks:
            feat = blk(feat)
        return x + self.head(feat)


In [ ]:
model = SpikingDehazeNet(dim=24, n_blocks=2, timesteps=4)
x = torch.randn(1, 3, 128, 128)
y = model(x)
print("输入:", x.shape, "输出:", y.shape)
print("参数量: %.2f M" % (sum(p.numel() for p in model.parameters()) / 1e6))
